In [2]:
# pip install pytest, ipytest, coverage, pytest-cov, 
import time
import json
import pytest
import ipytest
# ! will NEED PyTorch version 2.5+ --> for scaled_dot_product_attention(enable_gqa=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from einops import rearrange
from model_config import ModelConfig, AttentionConfig, FFNConfig
import training_engine as training_engine 
from tensor_tools import init_custom_weight
from inference_engine import advanced_inference, load_model_weights
from kv_cache import KV_cache


device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device:         {device}")
print(f"PyTorch:        {torch.__version__}")

if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"CUDA version:   {torch.version.cuda}")

    print(f"device capability: {torch.cuda.get_device_capability()}")
    print(f"flash_attention_available: {torch.backends.cuda.is_flash_attention_available()}")


Device:         cuda
PyTorch:        2.8.0+cu129
GPU:            NVIDIA GeForce GTX 1660 Ti
CUDA version:   12.9
device capability: (7, 5)
flash_attention_available: False


In [17]:

test_config = ModelConfig(

    model_name="test_models",
    tied_embeddings=False,  
    
    embed_dim=2**5,   # 32
    vocab_size=2**11,  # 2048
    num_layers=5,
    max_training_batch_size=2**4,   # 16
    max_context_window=2**6,   # 64    
    learning_rate=1e-3,

    attention=AttentionConfig(
        num_q_heads =2**0,  # 1 | 1   x   x    x
        num_kv_heads=2**0,  # 1 | 1   1   2+   x
        rope_theta  =10000.0, 
    ),
    
    ffn=FFNConfig(
        mlp_to_embed_expand_factor= 2**2,
        use_gate=True,
        num_experts= 2**0,
        num_experts_per_tkn=2**0,
    ),

    save_path       = "../models/test_models.pt",
    val_loss_history= "../models/test_models_val_loss_history.txt",

    checkpoint_dir= "../checkpoints/",
    checkpoint_every_steps= 250,
    keep_last_n_checkpoints= 3,

)

print(test_config)
print(test_config.head_dim)
print(test_config.ffn_hidden_dim)

ModelConfig(model_name='test_models', tied_embeddings=False, max_training_batch_size=16, embed_dim=32, vocab_size=2048, max_context_window=64, num_layers=5, learning_rate=0.001, attention=AttentionConfig(num_q_heads=1, num_kv_heads=1, rope_theta=10000.0), ffn=FFNConfig(use_gate=True, mlp_to_embed_expand_factor=4, num_experts=1, num_experts_per_tkn=1), save_path='../models/test_models.pt', val_loss_history='../models/test_models_val_loss_history.txt', checkpoint_dir='../checkpoints/', checkpoint_every_steps=250, keep_last_n_checkpoints=3)
32
128


In [3]:
# Initialize ipytest (Clears previous test registry)
ipytest.config(clean=True)


def add_one(x):
    return x + 1

# A basic test using plain assert
# def test_correct_addition():
#     assert add_one(3) == 4


# A parameterized test executing three distinct variations
@pytest.mark.parametrize("input_val, expected", [
    (1, 2),
    (5, 6),
    (10, 11),
    (19, 20),
])
def test_multiple_additions(input_val, expected):
    assert add_one(input_val) == expected


ipytest.run("-k test_multiple_additions")


======================================= test session starts =======================================
platform win32 -- Python 3.13.7, pytest-9.1.1, pluggy-1.6.0
rootdir: c:\Users\raman sh\Desktop\From Matrix to Mind\code
plugins: anyio-4.12.0, cov-7.1.0
collected 4 items

t_59c7b17834ac426b87741af5b0080e60.py ....                                                   [100%]

========================================= tests coverage ==========================================
_________________________ coverage: platform win32, python 3.13.7-final-0 _________________________

Name                                                               Stmts   Miss  Cover
--------------------------------------------------------------------------------------
C:\Users\raman sh\AppData\Local\Temp\ipykernel_7548\736305355.py       7      5    29%
--------------------------------------------------------------------------------------
TOTAL                                                                  7      

<ExitCode.OK: 0>

In [ ]:
# overfit-one-batch test: the loss should reach near zero.



In [ ]:
# RoPE test: scores should depend only on relative offset. seq of one tkn repeated, just differ in pos


In [ ]:
# cache equivalence test

@pytest.mark.parametrize("chunk", [1, 3, 8, 13])
@torch.no_grad()
def test_cache_matches_full_forward(chunk, W=8):
    cfg = test_config(max_context_window=W, num_q_heads=4, num_kv_heads=2)  # exercise GQA
    model = Level2_Model(cfg).eval()
    ids = torch.randint(0, cfg.vocab_size, (1, 40))
    T = ids.shape[1]
    i = torch.arange(T)
    mask = (i[None] <= i[:, None]) & (i[None] > i[:, None] - W)
    ref = model(ids, attn_mask=mask[None, None])

    cache = KV_cache(cfg, num_lanes=1, device="cpu")
    lane = torch.tensor([0])
    out = torch.cat([
       
            model(ids[:, s:s+chunk], 
            
            cache=cache, lane_ids=lane)
            for s in range(0, T, chunk)
        
        ], dim=1)

    
    torch.testing.assert_close(out, ref, atol=1e-5, rtol=1e-4)


In [ ]:
# attention variants: SHA / MQA / GQA / MHA

In [ ]:
# tied vs. untied embeddings


In [ ]:
# Report tokens/sec, peak memory, and cache memory per config, ideally with a profiler trace showing where time goes

In [ ]:
# Sliding window without attention sinks degrades past the window,